<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [6]</a>'.</span>

In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
proc_path = "C:\\Users\\ander\\OneDrive\\TCC_USP\\data_processed"
raw_path = "C:\\Users\\ander\\OneDrive\\TCC_USP\\data_raw"
interim_path = "C:\\Users\\ander\\OneDrive\\TCC_USP\\data_interim"
periodo_start = "2018-01-01"
periodo_end = "2025-01-31"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "01_preprocessing"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\content\drive\MyDrive\TCC_USP
PROC_PATH: C:\content\drive\MyDrive\TCC_USP\data_processed


In [3]:
# 2. Resolver caminhos
ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else \
       ("MeuDrive" if os.path.exists("/content/drive/MeuDrive") else "MyDrive")
BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
RAW_PATH = os.path.join(BASE_PATH, "data_raw")
PROC_PATH = os.path.join(BASE_PATH, "data_processed")
os.makedirs(PROC_PATH, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH :", RAW_PATH)
print("PROC_PATH:", PROC_PATH)

BASE_PATH: /content/drive/MyDrive/TCC_USP
RAW_PATH : /content/drive/MyDrive/TCC_USP\data_raw
PROC_PATH: /content/drive/MyDrive/TCC_USP\data_processed


In [4]:
# 3. Carregar IBOV
import pandas as pd, os
ibov_file = os.path.join(RAW_PATH, "ibovespa.csv")
assert os.path.exists(ibov_file), f"Arquivo nÃ£o encontrado: {ibov_file}"
df_ibov = pd.read_csv(ibov_file)
df_ibov["data"] = pd.to_datetime(df_ibov["data"])
print("IBOV:", df_ibov.shape)
display(df_ibov.head())

IBOV: (10, 2)


,data,close
0,2025-01-01,130000.00
1,2025-01-02,130222.22
2,2025-01-03,130444.44
3,2025-01-04,130666.67
4,2025-01-05,130888.89


In [5]:
# 4. Carregar NotÃ­cias
news_file = os.path.join(RAW_PATH, "noticias_exemplo.csv")
assert os.path.exists(news_file), f"Arquivo nÃ£o encontrado: {news_file}"
df_news = pd.read_csv(news_file)
df_news["data"] = pd.to_datetime(df_news["data"])
print("NEWS:", df_news.shape)
display(df_news.head())

NEWS: (3, 3)


,data,titulo,texto
0,2025-01-02,B3 sobe com otimismo,Mercado abre em alta com dados positivos
1,2025-01-05,DÃ³lar pressiona bolsa,Moeda americana sobe e traz cautela aos invest...
2,2025-01-08,Petrobras lidera altas,AÃ§Ãµes da Petrobras avanÃ§am apÃ³s anÃºncio d...


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [6]:
# 5. PrÃ©-processamento de texto
import re, nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_pt = set(stopwords.words("portuguese"))

def preprocess_text(t):
    t = re.sub(r"[^a-zA-ZÃ€-Ã¿\s]", " ", str(t)).lower()
    toks = [w for w in t.split() if w and w not in stop_pt]
    return " ".join(toks)

# Usar a coluna 'texto'
df_news["clean_text"] = df_news["texto"].apply(preprocess_text)

print("âœ… PrÃ©-processamento concluÃ­do!")
display(df_news[["texto", "clean_text"]].head())

ModuleNotFoundError: No module named 'nltk'

In [ ]:
# 6. Salvar processados
ibov_out  = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_out  = os.path.join(PROC_PATH, "noticias_clean.csv")
df_ibov.to_csv(ibov_out, index=False)
df_news.to_csv(news_out, index=False)

print("Salvo em data_processed:")
!ls -lh "{PROC_PATH}"